# Phase 2 - CMIP6 GCM Performance Evaluation

**Paper 1 | South Asia hydroclimatic zones**

This notebook implements proposal section **7.2 GCM Performance Evaluation** using the local files currently available in `output/cmip6/historical`. Those CMIP6 files are annual 1985-2014 grids, so this phase evaluates annual precipitation totals and annual mean Tmax/Tmin against ERA5-Land aggregated to the same annual scale for each completed hydroclimatic zone. The output is a model ranking and ensemble pool for Phase 3.

## Cell 1 - Imports and Configuration

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
import scipy

warnings.filterwarnings('ignore')

ROOT = (Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve())
OUTPUT_ROOT = ROOT / 'output'
PHASE_DIR = OUTPUT_ROOT / 'cmip6_eval'
FIG_DIR = PHASE_DIR / 'figures'
TABLE_DIR = PHASE_DIR / 'tables'
TS_DIR = PHASE_DIR / 'timeseries'

for d in [PHASE_DIR, FIG_DIR, TABLE_DIR, TS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

ZONES_FILE = OUTPUT_ROOT / 'zones' / 'hydroclimatic_zones_SA.nc'
ERA5_DIR = OUTPUT_ROOT / 'era5'
CMIP6_HIST_DIR = OUTPUT_ROOT / 'cmip6' / 'historical'

START_YEAR = 1985
END_YEAR = 2014
VARIABLES = ['pr', 'tasmax', 'tasmin']
ZONE_IDS = list(range(7))
ZONE_LABELS = {z: f'Z{z+1}' for z in ZONE_IDS}

MODELS = sorted([p.name for p in CMIP6_HIST_DIR.iterdir() if p.is_dir()])
print('Required packages loaded: numpy, pandas, xarray, matplotlib, scipy')
print(f'xarray version: {xr.__version__}')
print(f'Phase output: {PHASE_DIR}')
print(f'Models found: {len(MODELS)}')
print(MODELS)

## Cell 2 - Helper Functions

In [ ]:
def first_data_var(ds):
    if not ds.data_vars:
        raise ValueError('Dataset has no data variables')
    return list(ds.data_vars)[0]


def standardise_xy(da):
    rename = {}
    for cand in ['latitude', 'y']:
        if cand in da.coords or cand in da.dims:
            rename[cand] = 'lat'
    for cand in ['longitude', 'x']:
        if cand in da.coords or cand in da.dims:
            rename[cand] = 'lon'
    if rename:
        da = da.rename(rename)
    if 'lon' in da.coords and float(da.lon.max()) > 180:
        da = da.assign_coords(lon=((da.lon + 180) % 360) - 180).sortby('lon')
    if 'lat' in da.coords:
        da = da.sortby('lat')
    return da


def convert_units(da, var):
    units = str(da.attrs.get('units', '')).lower().replace(' ', '')
    out = da
    if var == 'pr':
        if any(token in units for token in ['kgm-2s-1', 'kg/m2/s', 'kgm**-2s**-1', 'm/s', 'ms-1']):
            out = da * 86400.0
            out.attrs['units'] = 'mm day-1'
        elif units in ['m', 'm/day', 'mday-1']:
            out = da * 1000.0
            out.attrs['units'] = 'mm day-1'
        else:
            out.attrs['units'] = da.attrs.get('units', 'mm day-1')
    else:
        if units in ['k', 'kelvin']:
            out = da - 273.15
            out.attrs['units'] = 'degC'
        else:
            sample_mean = float(da.isel(time=slice(0, min(30, da.sizes.get('time', 30)))).mean(skipna=True).compute())
            if sample_mean > 100:
                out = da - 273.15
                out.attrs['units'] = 'degC'
    return out


def load_da(path, var):
    ds = xr.open_dataset(path)
    name = var if var in ds.data_vars else first_data_var(ds)
    da = standardise_xy(ds[name])
    da = convert_units(da, var)
    keep_dims = [d for d in ['time', 'lat', 'lon'] if d in da.dims]
    return da.transpose(*keep_dims)


def annual_aggregate(da, var):
    if da.sizes.get('time', 0) == (END_YEAR - START_YEAR + 1):
        return da
    if var == 'pr':
        return da.resample(time='YS').sum(skipna=True)
    return da.resample(time='YS').mean(skipna=True)


def zone_mean_annual(da, zone_da, var):
    da = standardise_xy(da)
    if not (np.array_equal(da.lat.values, zone_da.lat.values) and np.array_equal(da.lon.values, zone_da.lon.values)):
        da = da.interp(lat=zone_da.lat, lon=zone_da.lon, method='nearest')
    annual = annual_aggregate(da, var)
    rows = []
    for z in ZONE_IDS:
        masked = annual.where(zone_da == z)
        ts = masked.mean(dim=('lat', 'lon'), skipna=True).to_series()
        tmp = ts.reset_index()
        tmp.columns = ['time', 'value']
        tmp['zone'] = ZONE_LABELS[z]
        tmp['variable'] = var
        rows.append(tmp)
    return pd.concat(rows, ignore_index=True)


def pearson_r(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    if mask.sum() < 3:
        return np.nan
    return float(np.corrcoef(obs[mask], sim[mask])[0, 1])


def kge(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    if mask.sum() < 3:
        return np.nan
    o = obs[mask]
    s = sim[mask]
    r = pearson_r(o, s)
    alpha = np.std(s) / (np.std(o) + 1e-12)
    beta = np.mean(s) / (np.mean(o) + 1e-12)
    return float(1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2))


def compute_metrics(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    if mask.sum() < 10:
        return {'n': int(mask.sum()), 'r': np.nan, 'std_ratio': np.nan, 'rmse': np.nan, 'bias': np.nan, 'kge': np.nan}
    o = obs[mask]
    s = sim[mask]
    return {
        'n': int(mask.sum()),
        'r': pearson_r(o, s),
        'std_ratio': float(np.std(s) / (np.std(o) + 1e-12)),
        'rmse': float(np.sqrt(np.mean((s - o)**2))),
        'bias': float(np.mean(s - o)),
        'kge': kge(o, s),
    }

## Cell 3 - Load Hydroclimatic Zones

In [ ]:
with xr.open_dataset(ZONES_FILE) as ds_z:
    zone_name = 'zone' if 'zone' in ds_z.data_vars else first_data_var(ds_z)
    zone_da = standardise_xy(ds_z[zone_name]).load()

print(zone_da)
print('Zone pixel counts:')
for z in ZONE_IDS:
    print(ZONE_LABELS[z], int((zone_da == z).sum()))

## Cell 4 - ERA5 Zone-Mean Annual Reference Series

In [ ]:
era5_csv = TS_DIR / 'era5_zone_annual.csv'

if era5_csv.exists():
    era5_annual = pd.read_csv(era5_csv, parse_dates=['time'])
else:
    parts = []
    for var in VARIABLES:
        for year in range(START_YEAR, END_YEAR + 1):
            f = ERA5_DIR / var / f'era5_land_{var}_{year}_clipped.nc'
            if not f.exists():
                print(f'Missing ERA5 file: {f}')
                continue
            print(f'ERA5 {var} {year}')
            da = load_da(f, var)
            parts.append(zone_mean_annual(da, zone_da, var))
            da.close()
    era5_annual = pd.concat(parts, ignore_index=True)
    era5_annual.to_csv(era5_csv, index=False)

era5_annual['source'] = 'ERA5-Land'
era5_annual.head(), era5_annual.shape

## Cell 5 - CMIP6 Historical Zone-Mean Annual Series

In [ ]:
cmip6_csv = TS_DIR / 'cmip6_historical_zone_annual.csv'

if cmip6_csv.exists():
    cmip6_annual = pd.read_csv(cmip6_csv, parse_dates=['time'])
else:
    parts = []
    missing = []
    for model in MODELS:
        model_dir = CMIP6_HIST_DIR / model
        for var in VARIABLES:
            f = model_dir / f'{model}_{var}_historical_{START_YEAR}_{END_YEAR}.nc'
            if not f.exists():
                missing.append(str(f.relative_to(ROOT)))
                continue
            print(f'CMIP6 {model} {var}')
            da = load_da(f, var)
            tmp = zone_mean_annual(da, zone_da, var)
            tmp['model'] = model
            parts.append(tmp)
            da.close()
    cmip6_annual = pd.concat(parts, ignore_index=True)
    cmip6_annual.to_csv(cmip6_csv, index=False)
    (TABLE_DIR / 'missing_cmip6_historical_files.txt').write_text('\n'.join(missing), encoding='utf-8')
    print(f'Missing CMIP6 files: {len(missing)}')

cmip6_annual.head(), cmip6_annual.shape

## Cell 6 - Evaluation Metrics by Model, Variable, and Zone

In [ ]:
records = []

era5_eval = era5_annual.rename(columns={'value': 'era5'})[['time', 'zone', 'variable', 'era5']]
for model, sub_model in cmip6_annual.groupby('model'):
    merged = sub_model.merge(era5_eval, on=['time', 'zone', 'variable'], how='inner')
    for (var, zone), sub in merged.groupby(['variable', 'zone']):
        m = compute_metrics(sub['era5'].to_numpy(), sub['value'].to_numpy())
        m.update({'model': model, 'variable': var, 'zone': zone})
        records.append(m)

metrics_df = pd.DataFrame(records)
metrics_df = metrics_df[['model', 'variable', 'zone', 'n', 'r', 'std_ratio', 'rmse', 'bias', 'kge']]
metrics_df.to_csv(TABLE_DIR / 'evaluation_metrics_by_model_variable_zone.csv', index=False)
metrics_df.head(), metrics_df.shape

## Cell 7 - Composite Skill Score and Ensemble Pool

In [ ]:
def minmax_good_high(s):
    s = s.astype(float)
    lo, hi = s.min(skipna=True), s.max(skipna=True)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return pd.Series(np.ones(len(s)), index=s.index)
    return (s - lo) / (hi - lo)


def minmax_good_low(s):
    s = s.astype(float)
    lo, hi = s.min(skipna=True), s.max(skipna=True)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return pd.Series(np.ones(len(s)), index=s.index)
    return 1 - (s - lo) / (hi - lo)


score_parts = []
for var, sub in metrics_df.groupby('variable'):
    s = sub.copy()
    s['kge_score'] = minmax_good_high(s['kge'])
    s['r_score'] = minmax_good_high(s['r'])
    s['rmse_score'] = minmax_good_low(s['rmse'])
    s['bias_score'] = minmax_good_low(s['bias'].abs())
    s['std_score'] = minmax_good_low((s['std_ratio'] - 1).abs())
    s['cell_score'] = (
        0.35 * s['kge_score'] +
        0.20 * s['r_score'] +
        0.20 * s['rmse_score'] +
        0.15 * s['bias_score'] +
        0.10 * s['std_score']
    )
    score_parts.append(s)

score_cells = pd.concat(score_parts, ignore_index=True)
ranking = (
    score_cells
    .groupby('model', as_index=False)
    .agg(
        composite_score=('cell_score', 'mean'),
        mean_kge=('kge', 'mean'),
        mean_r=('r', 'mean'),
        mean_rmse=('rmse', 'mean'),
        mean_abs_bias=('bias', lambda x: np.nanmean(np.abs(x))),
        n_metric_cells=('cell_score', 'count'),
    )
    .sort_values('composite_score', ascending=False)
)

cutoff = ranking['composite_score'].median()
ranking['in_ensemble_pool'] = ranking['composite_score'] >= cutoff
pool = ranking.loc[ranking['in_ensemble_pool'], 'model'].tolist()

score_cells.to_csv(TABLE_DIR / 'normalised_metric_scores.csv', index=False)
ranking.to_csv(TABLE_DIR / 'model_ranking.csv', index=False)
(TABLE_DIR / 'ensemble_pool.txt').write_text('\n'.join(pool), encoding='utf-8')

print('Composite score cutoff:', round(float(cutoff), 4))
print('Ensemble pool:', pool)
ranking

## Cell 8 - Diagnostic Figures

In [ ]:
mpl.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 140,
})

fig, ax = plt.subplots(figsize=(10, 5))
plot_rank = ranking.sort_values('composite_score', ascending=True)
colors = np.where(plot_rank['in_ensemble_pool'], '#2C7FB8', '#BDBDBD')
ax.barh(plot_rank['model'], plot_rank['composite_score'], color=colors)
ax.axvline(cutoff, color='black', linestyle='--', linewidth=1, label='median cutoff')
ax.set_xlabel('Composite skill score')
ax.set_ylabel('CMIP6 model')
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / 'model_ranking.png', bbox_inches='tight')
plt.show()

for metric in ['kge', 'bias', 'rmse', 'r']:
    fig, axes = plt.subplots(1, len(VARIABLES), figsize=(17, 5), sharey=True)
    for ax, var in zip(axes, VARIABLES):
        mat = metrics_df[metrics_df['variable'] == var].pivot_table(index='model', columns='zone', values=metric)
        # Reindex instead of .loc because CESM2 has precipitation only in the local archive.
        # Missing model-variable combinations remain NaN and plot as blank cells.
        mat = mat.reindex(ranking['model'])
        cmap = plt.get_cmap('RdBu_r' if metric == 'bias' else 'viridis').copy()
        cmap.set_bad('#F0F0F0')
        im = ax.imshow(np.ma.masked_invalid(mat.values), aspect='auto', cmap=cmap)
        ax.set_title(var)
        ax.set_xticks(range(len(mat.columns)))
        ax.set_xticklabels(mat.columns, rotation=45)
        ax.set_yticks(range(len(mat.index)))
        ax.set_yticklabels(mat.index)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(metric.upper())
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'{metric}_heatmaps.png', bbox_inches='tight')
    plt.show()

## Cell 9 - Completion Check

In [ ]:
expected = [
    TS_DIR / 'era5_zone_annual.csv',
    TS_DIR / 'cmip6_historical_zone_annual.csv',
    TABLE_DIR / 'evaluation_metrics_by_model_variable_zone.csv',
    TABLE_DIR / 'normalised_metric_scores.csv',
    TABLE_DIR / 'model_ranking.csv',
    TABLE_DIR / 'ensemble_pool.txt',
    FIG_DIR / 'model_ranking.png',
    FIG_DIR / 'kge_heatmaps.png',
    FIG_DIR / 'bias_heatmaps.png',
    FIG_DIR / 'rmse_heatmaps.png',
    FIG_DIR / 'r_heatmaps.png',
]

for p in expected:
    print(('OK   ' if p.exists() else 'MISS ') + str(p.relative_to(ROOT)))

## Cell 10 - O2 Proposal Completion Setup

The original O2 notebook already produced annual-scale ERA5/CMIP6 zone time series, core metrics, heatmaps, and the ensemble pool. The following cells add the missing proposal-facing diagnostics without changing the existing outputs:

- Taylor diagrams from annual zone-mean time series.
- 5-fold temporal cross-validation.
- Bias-ratio diagnostics.
- Model suitability matrix by variable.
- O2 completion checklist and manuscript draft notes.


In [ ]:
# Robust setup for rerunning this section independently after earlier O2 cells.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

try:
    ROOT
except NameError:
    ROOT = (Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve())
    EVAL_DIR = ROOT / 'output' / 'cmip6_eval'
    TS_DIR = EVAL_DIR / 'timeseries'
    TABLE_DIR = EVAL_DIR / 'tables'
    FIG_DIR = EVAL_DIR / 'figures'
    for d in [TS_DIR, TABLE_DIR, FIG_DIR]:
        d.mkdir(parents=True, exist_ok=True)

VARIABLES = ['pr', 'tasmax', 'tasmin']
VAR_TITLES = {'pr': 'Precipitation', 'tasmax': 'Tmax', 'tasmin': 'Tmin'}
VAR_UNITS = {'pr': 'mm yr-1', 'tasmax': 'degC', 'tasmin': 'degC'}
ZONE_ORDER = [f'Z{i}' for i in range(1, 8)]

# Load saved outputs.
era5_annual = pd.read_csv(TS_DIR / 'era5_zone_annual.csv')
cmip6_annual = pd.read_csv(TS_DIR / 'cmip6_historical_zone_annual.csv')
metrics_df = pd.read_csv(TABLE_DIR / 'evaluation_metrics_by_model_variable_zone.csv')
ranking = pd.read_csv(TABLE_DIR / 'model_ranking.csv')
score_cells = pd.read_csv(TABLE_DIR / 'normalised_metric_scores.csv')

era5_annual['year'] = pd.to_datetime(era5_annual['time']).dt.year
cmip6_annual['year'] = pd.to_datetime(cmip6_annual['time']).dt.year
print('Loaded O2 annual time series and existing metrics.')
print('ERA5:', era5_annual.shape, 'CMIP6:', cmip6_annual.shape, 'metrics:', metrics_df.shape)


## Cell 11 - Bias Ratio Diagnostics


In [ ]:
# Bias ratio is especially meaningful for precipitation. For temperature, it is saved as an auxiliary diagnostic only;
# mean bias remains the primary temperature bias metric.
ratio_rows = []
era5_eval = era5_annual.rename(columns={'value': 'era5'})[['year', 'zone', 'variable', 'era5']]
for model, sub_model in cmip6_annual.groupby('model'):
    merged = sub_model.merge(era5_eval, on=['year', 'zone', 'variable'], how='inner')
    for (var, zone), sub in merged.groupby(['variable', 'zone']):
        obs_mean = float(np.nanmean(sub['era5']))
        sim_mean = float(np.nanmean(sub['value']))
        ratio_rows.append({
            'model': model,
            'variable': var,
            'zone': zone,
            'obs_mean': obs_mean,
            'model_mean': sim_mean,
            'bias_ratio': sim_mean / obs_mean if abs(obs_mean) > 1e-10 else np.nan,
            'percent_bias': 100 * (sim_mean - obs_mean) / obs_mean if abs(obs_mean) > 1e-10 else np.nan,
            'note': 'Bias ratio is primary for precipitation; temperature ratio is auxiliary only.' if var != 'pr' else 'Precipitation bias ratio.',
        })

bias_ratio_df = pd.DataFrame(ratio_rows)
metrics_ext = metrics_df.merge(bias_ratio_df[['model', 'variable', 'zone', 'obs_mean', 'model_mean', 'bias_ratio', 'percent_bias']], on=['model', 'variable', 'zone'], how='left')
bias_ratio_df.to_csv(TABLE_DIR / 'bias_ratio_by_model_variable_zone.csv', index=False)
metrics_ext.to_csv(TABLE_DIR / 'evaluation_metrics_with_bias_ratio.csv', index=False)
print('[OK]', TABLE_DIR / 'bias_ratio_by_model_variable_zone.csv')
print('[OK]', TABLE_DIR / 'evaluation_metrics_with_bias_ratio.csv')
bias_ratio_df.head()


## Cell 12 - Taylor Diagrams


In [ ]:

# Taylor diagrams with compact model codes to avoid label overlap.
model_order = ranking.sort_values('composite_score', ascending=False)['model'].tolist()
model_code_map = {m: f'M{i+1:02d}' for i, m in enumerate(model_order)}
model_code_key = pd.DataFrame({'model': model_order, 'model_code': [model_code_map[m] for m in model_order]})
model_code_key.to_csv(TABLE_DIR / 'o2_model_code_key.csv', index=False)


def taylor_stats_for_variable(var):
    obs_df = era5_annual[era5_annual['variable'] == var][['year', 'zone', 'value']].rename(columns={'value': 'era5'})
    rows = []
    ref_values = obs_df['era5'].to_numpy()
    ref_std = float(np.nanstd(ref_values))
    for model, mdf in cmip6_annual[cmip6_annual['variable'] == var].groupby('model'):
        merged = mdf.merge(obs_df, on=['year', 'zone'], how='inner')
        mask = np.isfinite(merged['era5']) & np.isfinite(merged['value'])
        if mask.sum() < 8:
            rows.append({'model': model, 'model_code': model_code_map.get(model, model), 'std_ratio': np.nan, 'r': np.nan, 'centered_rmse_norm': np.nan, 'n': int(mask.sum())})
            continue
        obs = merged.loc[mask, 'era5'].to_numpy()
        sim = merged.loc[mask, 'value'].to_numpy()
        sim_std = np.nanstd(sim)
        obs_std = np.nanstd(obs)
        r = np.corrcoef(obs, sim)[0, 1]
        centered_rmse = np.sqrt(np.mean(((sim - sim.mean()) - (obs - obs.mean())) ** 2))
        rows.append({
            'model': model,
            'model_code': model_code_map.get(model, model),
            'std_ratio': float(sim_std / (obs_std + 1e-10)),
            'r': float(r),
            'centered_rmse_norm': float(centered_rmse / (obs_std + 1e-10)),
            'n': int(mask.sum()),
        })
    return pd.DataFrame(rows), ref_std


def draw_taylor(ax, stats, title, zoom=False):
    finite = stats[np.isfinite(stats['std_ratio']) & np.isfinite(stats['r'])].copy()
    rmax = 1.25 if zoom else max(1.6, float(finite['std_ratio'].max()) * 1.15 if not finite.empty else 1.6)
    theta = np.linspace(0, np.pi / 2, 240)
    for rr in np.arange(0.25, rmax + 0.25, 0.25):
        ax.plot(rr * np.cos(theta), rr * np.sin(theta), color='0.86', lw=0.6, zorder=0)
    for corr in [0.0, 0.2, 0.4, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]:
        th = np.arccos(corr)
        ax.plot([0, rmax*np.cos(th)], [0, rmax*np.sin(th)], color='0.78', lw=0.5, ls='--', zorder=0)
        if (not zoom) or corr >= 0.7:
            ax.text(rmax*1.035*np.cos(th), rmax*1.035*np.sin(th), f'{corr:g}', fontsize=7, ha='center', va='center')
    for rms in [0.25, 0.5, 0.75, 1.0, 1.25]:
        th = np.linspace(0, np.pi, 300)
        x = 1 + rms * np.cos(th)
        y = rms * np.sin(th)
        m = (x >= 0) & (y >= 0) & (x <= rmax) & (y <= rmax)
        ax.plot(x[m], y[m], color='#2E7D32', lw=0.6, ls=':', alpha=0.6, zorder=0)
    ax.plot(1, 0, 'k*', ms=14, label='ERA5')
    cmap = plt.get_cmap('tab20')
    for i, row in finite.sort_values('model_code').reset_index(drop=True).iterrows():
        th = np.arccos(np.clip(row['r'], 0, 1))
        x = row['std_ratio'] * np.cos(th)
        y = row['std_ratio'] * np.sin(th)
        ax.plot(x, y, 'o', ms=8, color=cmap(i % 20), markeredgecolor='k', markeredgewidth=0.4)
        ax.annotate(row['model_code'], (x, y), xytext=(4, 4), textcoords='offset points', fontsize=7, fontweight='bold')
    if zoom:
        ax.set_xlim(0.55, 1.08)
        ax.set_ylim(0, 0.48)
    else:
        ax.set_xlim(0, rmax)
        ax.set_ylim(0, rmax)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('Normalised standard deviation', fontweight='bold')
    ax.set_ylabel('Normalised standard deviation', fontweight='bold')
    ax.set_title(title, loc='left', fontweight='bold')
    ax.text(0.98, 0.04, 'Correlation', transform=ax.transAxes, ha='right', fontsize=8, color='0.35')

mpl.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10, 'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})
taylor_rows = []

# Full Taylor diagram for method documentation.
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
for ax, var, letter in zip(axes, VARIABLES, list('ABC')):
    stats, ref_std = taylor_stats_for_variable(var)
    stats['variable'] = var
    stats['era5_ref_std'] = ref_std
    taylor_rows.append(stats)
    draw_taylor(ax, stats, f'{letter}. {VAR_TITLES[var]}', zoom=False)
fig.suptitle('Taylor diagrams for annual CMIP6 historical simulations against ERA5, 1985-2014', fontsize=14, fontweight='bold')
out_png = FIG_DIR / 'taylor_diagrams_annual.png'
out_pdf = FIG_DIR / 'taylor_diagrams_annual.pdf'
fig.savefig(out_png, dpi=300, bbox_inches='tight')
fig.savefig(out_pdf, bbox_inches='tight')
plt.show()

# Zoomed Taylor diagram for manuscript readability.
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), constrained_layout=True)
for ax, var, letter in zip(axes, VARIABLES, list('ABC')):
    stats = [df for df in taylor_rows if df['variable'].iloc[0] == var][0]
    draw_taylor(ax, stats, f'{letter}. {VAR_TITLES[var]}', zoom=True)
fig.suptitle('Zoomed Taylor diagrams for annual CMIP6 historical simulations against ERA5', fontsize=14, fontweight='bold')
out_zoom_png = FIG_DIR / 'taylor_diagrams_annual_zoomed.png'
out_zoom_pdf = FIG_DIR / 'taylor_diagrams_annual_zoomed.pdf'
fig.savefig(out_zoom_png, dpi=300, bbox_inches='tight')
fig.savefig(out_zoom_pdf, bbox_inches='tight')
plt.show()

taylor_df = pd.concat(taylor_rows, ignore_index=True)
taylor_df.to_csv(TABLE_DIR / 'taylor_diagram_statistics_annual.csv', index=False)
print('[OK]', out_png.relative_to(ROOT))
print('[OK]', out_zoom_png.relative_to(ROOT))
print('[OK]', TABLE_DIR / 'taylor_diagram_statistics_annual.csv')
print('[OK]', TABLE_DIR / 'o2_model_code_key.csv')


## Cell 13 - Five-Fold Temporal Cross-Validation


In [ ]:

from sklearn.model_selection import KFold

# Fold-specific metrics: annual folds have only six test years per zone, so the earlier full-period
# minimum-sample threshold is intentionally not reused here.
def compute_metrics_cv(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    if mask.sum() < 3:
        return {'n': int(mask.sum()), 'r': np.nan, 'std_ratio': np.nan, 'rmse': np.nan, 'bias': np.nan, 'kge': np.nan}
    o = obs[mask]
    s = sim[mask]
    r = np.corrcoef(o, s)[0, 1] if len(o) > 1 and np.nanstd(o) > 0 and np.nanstd(s) > 0 else np.nan
    std_ratio = np.nanstd(s) / (np.nanstd(o) + 1e-10)
    beta = np.nanmean(s) / (np.nanmean(o) + 1e-10)
    kge = 1 - np.sqrt((r - 1)**2 + (std_ratio - 1)**2 + (beta - 1)**2) if np.isfinite(r) else np.nan
    return {
        'n': int(mask.sum()),
        'r': float(r) if np.isfinite(r) else np.nan,
        'std_ratio': float(std_ratio),
        'rmse': float(np.sqrt(np.mean((s - o) ** 2))),
        'bias': float(np.mean(s - o)),
        'kge': float(kge) if np.isfinite(kge) else np.nan,
    }

# Ensure era5_eval exists if this cell is rerun alone.
era5_eval = era5_annual.rename(columns={'value': 'era5'})[['year', 'zone', 'variable', 'era5']]

cv_rows = []
kf = KFold(n_splits=5, shuffle=False)
for model, sub_model in cmip6_annual.groupby('model'):
    merged = sub_model.merge(era5_eval, on=['year', 'zone', 'variable'], how='inner')
    for (var, zone), sub in merged.groupby(['variable', 'zone']):
        sub = sub.sort_values('year').reset_index(drop=True)
        years = np.array(sorted(sub['year'].unique()))
        if len(years) < 10:
            continue
        for fold_id, (_, test_idx) in enumerate(kf.split(years), start=1):
            test_years = years[test_idx]
            test = sub[sub['year'].isin(test_years)]
            m = compute_metrics_cv(test['era5'].to_numpy(), test['value'].to_numpy())
            m.update({'model': model, 'variable': var, 'zone': zone, 'fold': fold_id, 'test_years': f'{test_years.min()}-{test_years.max()}'})
            cv_rows.append(m)

cv_df = pd.DataFrame(cv_rows)
cv_df = cv_df[['model', 'variable', 'zone', 'fold', 'test_years', 'n', 'r', 'std_ratio', 'rmse', 'bias', 'kge']]
cv_summary = (
    cv_df.groupby(['model', 'variable', 'zone'], as_index=False)
    .agg(
        cv_mean_kge=('kge', 'mean'), cv_std_kge=('kge', 'std'),
        cv_mean_r=('r', 'mean'), cv_std_r=('r', 'std'),
        cv_mean_rmse=('rmse', 'mean'), cv_std_rmse=('rmse', 'std'),
        cv_mean_bias=('bias', 'mean'), cv_std_bias=('bias', 'std'),
        n_folds=('fold', 'count'),
    )
)
cv_model_summary = (
    cv_summary.groupby('model', as_index=False)
    .agg(
        cv_overall_mean_kge=('cv_mean_kge', 'mean'),
        cv_overall_std_kge=('cv_mean_kge', 'std'),
        cv_overall_mean_r=('cv_mean_r', 'mean'),
        cv_overall_mean_rmse=('cv_mean_rmse', 'mean'),
        n_cv_cells=('cv_mean_kge', 'count'),
    )
    .sort_values('cv_overall_mean_kge', ascending=False)
)

cv_df.to_csv(TABLE_DIR / 'kfold_temporal_cv_metrics_by_fold.csv', index=False)
cv_summary.to_csv(TABLE_DIR / 'kfold_temporal_cv_summary_by_model_variable_zone.csv', index=False)
cv_model_summary.to_csv(TABLE_DIR / 'kfold_temporal_cv_model_summary.csv', index=False)
print('[OK]', TABLE_DIR / 'kfold_temporal_cv_metrics_by_fold.csv')
print('[OK]', TABLE_DIR / 'kfold_temporal_cv_model_summary.csv')
cv_model_summary.head(10)


## Cell 14 - Model Suitability Matrix


In [ ]:
# Variable-specific suitability from existing normalised metric cell scores.
var_scores = (
    score_cells.groupby(['model', 'variable'], as_index=False)
    .agg(variable_skill_score=('cell_score', 'mean'), mean_kge=('kge', 'mean'), mean_r=('r', 'mean'), mean_rmse=('rmse', 'mean'), mean_abs_bias=('bias', lambda x: np.nanmean(np.abs(x))), n_zones=('zone', 'nunique'))
)
var_scores['variable_rank'] = var_scores.groupby('variable')['variable_skill_score'].rank(ascending=False, method='min').astype(int)
var_scores['variable_median_score'] = var_scores.groupby('variable')['variable_skill_score'].transform('median')
var_scores['selected_for_variable'] = var_scores['variable_skill_score'] >= var_scores['variable_median_score']

# Wide suitability matrix for quick manuscript/table use.
suitability = var_scores.pivot_table(index='model', columns='variable', values='selected_for_variable', aggfunc='first')
suitability = suitability.reindex(columns=VARIABLES)
suitability = suitability.fillna(False).astype(bool)
suitability['full_tas_pr_available'] = suitability[['pr', 'tasmax', 'tasmin']].all(axis=1)
suitability['precip_only_or_partial'] = suitability['pr'] & ~suitability['full_tas_pr_available']
suitability = suitability.reset_index()
suitability = suitability.merge(ranking[['model', 'composite_score', 'in_ensemble_pool', 'n_metric_cells']], on='model', how='left')

def recommendation(row):
    if row['full_tas_pr_available'] and row['in_ensemble_pool']:
        return 'full-variable ensemble candidate'
    if row['pr'] and (not row['tasmax'] or not row['tasmin']):
        return 'precipitation-only or partial-use candidate'
    if row['in_ensemble_pool']:
        return 'selected by composite score but check variable coverage'
    return 'not selected for primary ensemble'

suitability['recommendation'] = suitability.apply(recommendation, axis=1)
var_scores.to_csv(TABLE_DIR / 'variable_specific_model_skill_scores.csv', index=False)
suitability.to_csv(TABLE_DIR / 'model_suitability_matrix.csv', index=False)
print('[OK]', TABLE_DIR / 'variable_specific_model_skill_scores.csv')
print('[OK]', TABLE_DIR / 'model_suitability_matrix.csv')
suitability.sort_values('composite_score', ascending=False)


## Cell 15 - O2 Completion Checklist and Summary Draft


In [ ]:
expected_o2 = [
    TABLE_DIR / 'evaluation_metrics_by_model_variable_zone.csv',
    TABLE_DIR / 'evaluation_metrics_with_bias_ratio.csv',
    TABLE_DIR / 'bias_ratio_by_model_variable_zone.csv',
    TABLE_DIR / 'normalised_metric_scores.csv',
    TABLE_DIR / 'model_ranking.csv',
    TABLE_DIR / 'ensemble_pool.txt',
    TABLE_DIR / 'taylor_diagram_statistics_annual.csv',
    TABLE_DIR / 'kfold_temporal_cv_metrics_by_fold.csv',
    TABLE_DIR / 'kfold_temporal_cv_summary_by_model_variable_zone.csv',
    TABLE_DIR / 'kfold_temporal_cv_model_summary.csv',
    TABLE_DIR / 'variable_specific_model_skill_scores.csv',
    TABLE_DIR / 'model_suitability_matrix.csv',
    FIG_DIR / 'model_ranking.png',
    FIG_DIR / 'kge_heatmaps.png',
    FIG_DIR / 'bias_heatmaps.png',
    FIG_DIR / 'rmse_heatmaps.png',
    FIG_DIR / 'r_heatmaps.png',
    FIG_DIR / 'taylor_diagrams_annual.png',
]
checklist = pd.DataFrame({'output': [str(p.relative_to(ROOT)) for p in expected_o2], 'exists': [p.exists() for p in expected_o2]})
checklist.to_csv(TABLE_DIR / 'o2_completion_checklist.csv', index=False)

pool = (TABLE_DIR / 'ensemble_pool.txt').read_text(encoding='utf-8').strip().splitlines()
top = ranking.sort_values('composite_score', ascending=False).iloc[0]
summary = f"""# O2 CMIP6 GCM Performance Evaluation - Completion Summary

## Purpose

This objective evaluates historical CMIP6 model skill for annual precipitation, Tmax, and Tmin across the seven hydroclimatic zones using ERA5-Land as the reference over 1985-2014.

## Completed Diagnostics

- KGE, RMSE, mean bias, Pearson correlation, and standard-deviation ratio by model, variable, and zone.
- Bias-ratio and percent-bias diagnostics.
- Taylor diagram statistics and annual Taylor diagrams.
- Five-fold temporal cross-validation by model, variable, and zone.
- Composite skill scoring and model ranking.
- Variable-specific model suitability matrix.
- Ensemble pool selection.

## Key Result

The top-ranked model by composite skill score is {top['model']} with a score of {top['composite_score']:.3f}. The selected preliminary ensemble pool is: {', '.join(pool)}.

## Important Caveat

The current O2 implementation uses annual zone-mean time series because those are the local CMIP6 products available in this project. This is consistent with the saved data products, but if the manuscript claims monthly Taylor diagrams or daily-scale validation, those claims should be revised or the monthly/daily evaluation should be rerun explicitly.
"""
(TABLE_DIR / 'o2_completion_summary.md').write_text(summary, encoding='utf-8')
print(checklist.to_string(index=False))
print('\nAll complete:', checklist['exists'].all())
print(TABLE_DIR / 'o2_completion_summary.md')


In [ ]:
# Cell 16 - Readable Taylor-Statistics Visualisations for Manuscript
# This cell is robust to running from either the project root or the notebook folder.
# It creates two clear alternatives to the compressed classical Taylor diagram.

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


def find_project_root():
    """Find the project root containing output/cmip6_eval/tables."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for cand in candidates:
        if (cand / 'output' / 'cmip6_eval' / 'tables').exists():
            return cand
    # Fallback for this project if the notebook is opened from a subfolder.
    for cand in [Path('..').resolve(), Path('../..').resolve()]:
        if (cand / 'output' / 'cmip6_eval' / 'tables').exists():
            return cand
    raise FileNotFoundError('Could not find project root containing output/cmip6_eval/tables')

PROJECT_ROOT = find_project_root()
OUT_ROOT = PROJECT_ROOT / 'output' / 'cmip6_eval'
TABLE_DIR = OUT_ROOT / 'tables'
FIG_DIR = OUT_ROOT / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'[INFO] Project root: {PROJECT_ROOT}')

# Read Taylor statistics. If the table is missing, rebuild it from annual zone means.
taylor_path = TABLE_DIR / 'taylor_diagram_statistics_annual.csv'
code_key_path = TABLE_DIR / 'o2_model_code_key.csv'

if not taylor_path.exists():
    print('[INFO] Taylor statistics table missing; rebuilding from annual zone means.')
    era5_eval = pd.read_csv(OUT_ROOT / 'timeseries' / 'era5_zone_annual.csv')
    cmip_eval = pd.read_csv(OUT_ROOT / 'timeseries' / 'cmip6_historical_zone_annual.csv')
    ranking = pd.read_csv(TABLE_DIR / 'model_ranking.csv')
    ranked_models = ranking['model'].drop_duplicates().tolist()
    model_codes = {model: f'M{i:02d}' for i, model in enumerate(ranked_models, start=1)}
    rows = []
    for var in ['pr', 'tasmax', 'tasmin']:
        obs = era5_eval[era5_eval['variable'] == var][['year', 'zone', 'value']].rename(columns={'value': 'obs'})
        for model in sorted(cmip_eval.loc[cmip_eval['variable'] == var, 'model'].dropna().unique()):
            sim = cmip_eval[(cmip_eval['variable'] == var) & (cmip_eval['model'] == model)][['year', 'zone', 'value']].rename(columns={'value': 'sim'})
            merged = obs.merge(sim, on=['year', 'zone']).dropna()
            if len(merged) < 3:
                continue
            o = merged['obs'].to_numpy(dtype=float)
            s = merged['sim'].to_numpy(dtype=float)
            ref_std = np.std(o, ddof=1)
            sim_std = np.std(s, ddof=1)
            r = np.corrcoef(o, s)[0, 1] if ref_std > 0 and sim_std > 0 else np.nan
            centered_rmse = np.sqrt(np.mean(((s - s.mean()) - (o - o.mean())) ** 2))
            rows.append({
                'model': model,
                'model_code': model_codes.get(model, model),
                'std_ratio': sim_std / ref_std if ref_std > 0 else np.nan,
                'r': r,
                'centered_rmse_norm': centered_rmse / ref_std if ref_std > 0 else np.nan,
                'n': len(merged),
                'variable': var,
                'era5_ref_std': ref_std,
            })
    taylor = pd.DataFrame(rows)
    taylor.to_csv(taylor_path, index=False)
    pd.DataFrame([{'model_code': v, 'model': k} for k, v in model_codes.items()]).to_csv(code_key_path, index=False)
else:
    taylor = pd.read_csv(taylor_path)

code_key = pd.read_csv(code_key_path) if code_key_path.exists() else None

var_order = ['pr', 'tasmax', 'tasmin']
var_labels = {'pr': 'Precipitation', 'tasmax': 'Tmax', 'tasmin': 'Tmin'}
colors = {'pr': '#2f80b7', 'tasmax': '#c74d3f', 'tasmin': '#2f8f55'}

# Figure 1: readable Taylor-statistics scatter.
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.8), constrained_layout=True)
fig.suptitle('CMIP6 Historical Skill Relative to ERA5 (1985-2014)', fontsize=15, fontweight='bold')

for panel, (ax, var) in enumerate(zip(axes, var_order), start=1):
    sub = taylor[taylor['variable'] == var].copy().sort_values(['r', 'std_ratio'])
    if sub.empty:
        ax.axis('off')
        continue
    x = sub['r'].astype(float).to_numpy()
    y = sub['std_ratio'].astype(float).to_numpy()
    xpad = max((x.max() - x.min()) * 0.35, 0.004)
    ypad = max((y.max() - y.min()) * 0.45, 0.015)
    xmin = max(0.0, x.min() - xpad)
    xmax = min(1.005, max(1.0, x.max() + xpad))
    ymin = max(0.0, y.min() - ypad)
    ymax = max(1.02, y.max() + ypad)

    ax.scatter(sub['r'], sub['std_ratio'], s=78, color=colors[var], edgecolor='black', linewidth=0.7, zorder=3)
    ax.scatter([1.0], [1.0], marker='*', s=170, color='black', zorder=4)
    offsets = np.linspace(-0.012, 0.012, len(sub))
    for i, (_, row) in enumerate(sub.reset_index(drop=True).iterrows()):
        dx = offsets[i] * (xmax - xmin)
        dy = (0.018 if i % 2 == 0 else -0.022) * (ymax - ymin)
        ax.annotate(
            str(row['model_code']), xy=(row['r'], row['std_ratio']),
            xytext=(row['r'] + dx, row['std_ratio'] + dy), textcoords='data',
            ha='center', va='center', fontsize=8.5, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='0.65', lw=0.45, alpha=0.88),
            arrowprops=dict(arrowstyle='-', color='0.55', lw=0.45, shrinkA=0, shrinkB=5), zorder=5,
        )
    ax.axvline(1.0, color='0.25', linestyle='--', linewidth=1.0)
    ax.axhline(1.0, color='0.25', linestyle='--', linewidth=1.0)
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.grid(True, linestyle=':', linewidth=0.7, alpha=0.65)
    ax.set_title(f"{chr(64 + panel)}. {var_labels[var]}", loc='left', fontsize=12, fontweight='bold')
    ax.set_xlabel('Correlation with ERA5 (r)', fontsize=10.5, fontweight='bold')
    if panel == 1:
        ax.set_ylabel('Normalised standard deviation', fontsize=10.5, fontweight='bold')
    ax.tick_params(axis='both', labelsize=9.5, width=1.0)
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontweight('bold')
    for spine in ax.spines.values():
        spine.set_linewidth(1.1)

if code_key is not None and not code_key.empty:
    legend_text = '\n'.join(f"{r.model_code}: {r.model}" for r in code_key.itertuples(index=False))
    fig.text(0.995, 0.49, legend_text, ha='right', va='center', fontsize=8.2, family='monospace',
             bbox=dict(boxstyle='round,pad=0.35', fc='white', ec='0.45', lw=0.7))

fig.text(0.01, 0.01, 'Note: readable companion to the classical Taylor diagram; clustering indicates similar annual zone-mean skill among models.', fontsize=8.5, color='0.25')

for suffix in ['png', 'pdf']:
    path = FIG_DIR / f'taylor_skill_scatter_annual_readable.{suffix}'
    fig.savefig(path, dpi=350 if suffix == 'png' else None, bbox_inches='tight')
    print(f'[OK] {path}')
plt.show()

# Figure 2: alternative component heatmap. This is clearer than Taylor geometry for manuscript readers.
metric_labels = {
    'r': 'Correlation (r)',
    'std_ratio': 'Std. ratio',
    'centered_rmse_norm': 'Centred RMSE / ERA5 std.'
}
models = code_key['model'].tolist() if code_key is not None and not code_key.empty else sorted(taylor['model'].unique())
fig, axes = plt.subplots(1, 3, figsize=(13.5, max(4.8, 0.36 * len(models))), constrained_layout=True)
fig.suptitle('Taylor Skill Components by Model and Variable', fontsize=15, fontweight='bold')

for ax, metric in zip(axes, ['r', 'std_ratio', 'centered_rmse_norm']):
    mat_rows = []
    ylabels = []
    for model in models:
        code = code_key.loc[code_key['model'] == model, 'model_code'].iloc[0] if code_key is not None and model in code_key['model'].values else model
        ylabels.append(f'{code} {model}')
        row = []
        for var in var_order:
            vals = taylor[(taylor['model'] == model) & (taylor['variable'] == var)][metric]
            row.append(float(vals.iloc[0]) if len(vals) else np.nan)
        mat_rows.append(row)
    mat = np.array(mat_rows, dtype=float)
    cmap = 'viridis_r' if metric == 'centered_rmse_norm' else 'viridis'
    im = ax.imshow(mat, aspect='auto', cmap=cmap)
    ax.set_title(metric_labels[metric], fontsize=11.5, fontweight='bold')
    ax.set_xticks(range(len(var_order)))
    ax.set_xticklabels([var_labels[v] for v in var_order], rotation=25, ha='right', fontweight='bold')
    ax.set_yticks(range(len(ylabels)))
    ax.set_yticklabels(ylabels if metric == 'r' else [''] * len(ylabels), fontsize=8.5, fontweight='bold')
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            if np.isfinite(mat[i, j]):
                ax.text(j, i, f'{mat[i, j]:.2f}', ha='center', va='center', fontsize=7.5, fontweight='bold', color='white' if metric != 'std_ratio' else 'black')
    cbar = fig.colorbar(im, ax=ax, shrink=0.88)
    cbar.ax.tick_params(labelsize=8)

for suffix in ['png', 'pdf']:
    path = FIG_DIR / f'taylor_skill_components_heatmap.{suffix}'
    fig.savefig(path, dpi=350 if suffix == 'png' else None, bbox_inches='tight')
    print(f'[OK] {path}')
plt.show()
